In [1]:
# ============================================================
# TESTE — SCENARIO AGENT OPENAI + CISEI DOMAIN CONTEXT
#
# Célula 1 — Ambiente e imports
# ============================================================

from pathlib import Path
import sys
import os

# ------------------------------------------------------------
# Localização do projeto
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/jovyan/work/planapp-mcp")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.version)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT existe:", PROJECT_ROOT.exists())

# ------------------------------------------------------------
# Imports do projeto
# ------------------------------------------------------------

from cisei_planning_sdk.project import PlanningProject

from scenario_agent_openai import ScenarioAgentOpenAI

from scenario_domain.providers import (
    CISEIPluginDomainContextProvider,
)

print()
print("IMPORTS OK")
print("PlanningProject:", PlanningProject)
print("ScenarioAgentOpenAI:", ScenarioAgentOpenAI)
print(
    "CISEIPluginDomainContextProvider:",
    CISEIPluginDomainContextProvider,
)

Python: 3.13.12 | packaged by conda-forge | (main, Feb  5 2026, 05:53:46) [GCC 14.3.0]
PROJECT_ROOT: /home/jovyan/work/planapp-mcp
PROJECT_ROOT existe: True

IMPORTS OK
PlanningProject: <class 'cisei_planning_sdk.project.PlanningProject'>
ScenarioAgentOpenAI: <class 'scenario_agent_openai.ScenarioAgentOpenAI'>
CISEIPluginDomainContextProvider: <class 'scenario_domain.providers.cisei_plugin.CISEIPluginDomainContextProvider'>


In [2]:
# ============================================================
# Célula 2 — CISEI Plugin Domain Context
# ============================================================

PLUGIN_ROOT = PROJECT_ROOT / "cisei-planning-engineering"

print("PLUGIN_ROOT:", PLUGIN_ROOT)
print("PLUGIN_ROOT existe:", PLUGIN_ROOT.exists())

# ------------------------------------------------------------
# Criar o provider
# ------------------------------------------------------------

domain_provider = CISEIPluginDomainContextProvider(
    PLUGIN_ROOT
)

print()
print("PROVIDER OK")
print("provider_id:", domain_provider.provider_id)
print("provider_version:", domain_provider.provider_version)
print("context_version:", domain_provider.context_version)

# ------------------------------------------------------------
# Seções disponíveis
# ------------------------------------------------------------

sections = domain_provider.available_sections()

print()
print("SEÇÕES DISPONÍVEIS:", len(sections))

for section in sections:
    print(" -", section)

# ------------------------------------------------------------
# Capacidades
# ------------------------------------------------------------

print()
print("CAPABILITIES:")

for key, value in domain_provider.capabilities().items():
    print(f" - {key}: {value}")

PLUGIN_ROOT: /home/jovyan/work/planapp-mcp/cisei-planning-engineering
PLUGIN_ROOT existe: True

PROVIDER OK
provider_id: cisei_plugin
provider_version: 0.4.0
context_version: 1

SEÇÕES DISPONÍVEIS: 11
 - agent_instructions
 - current_architecture
 - workspace_and_sdk
 - scenario_format
 - service_api_and_results
 - assistant_workflow
 - design_gaps_and_decisions
 - canonical_cases
 - usage_guide
 - mcp_interface_requirements
 - exploration_workflow

CAPABILITIES:
 - domain_context: True
 - section_selection: True
 - provenance: True
 - versioning: True
 - scenario_defaults: False
 - scenario_mutation: False
 - scenario_validation: False
 - planning_execution: False
 - mcp_execution: False
 - llm_execution: False


In [3]:
# ============================================================
# Célula 3 — Domain Context selecionado para o agente
# ============================================================

DEFAULT_DOMAIN_CONTEXT_SECTIONS = (
    "scenario_format",
    "assistant_workflow",
    "design_gaps_and_decisions",
)

domain_context = domain_provider.get_context(
    sections=DEFAULT_DOMAIN_CONTEXT_SECTIONS
)

print("DOMAIN CONTEXT CARREGADO")
print()
print("provider_id:", domain_context.provider_id)
print("provider_version:", domain_context.provider_version)
print("context_version:", domain_context.context_version)
print("sections:", len(domain_context.sections))

print()
print("SEÇÕES SELECIONADAS:")

for section in domain_context.sections:
    print(
        f" - {section.name}"
        f" | classification={section.classification}"
        f" | source={section.source}"
    )

print()
print("METADATA:")

for key, value in domain_context.metadata.items():
    print(f" - {key}: {value}")

DOMAIN CONTEXT CARREGADO

provider_id: cisei_plugin
provider_version: 0.4.0
context_version: 1
sections: 3

SEÇÕES SELECIONADAS:
 - scenario_format | classification=Implemented | source=references/03_SCENARIO_FORMAT.md
 - assistant_workflow | classification=Proposal | source=references/05_ASSISTANT_WORKFLOW.md
 - design_gaps_and_decisions | classification=Decision | source=references/06_DESIGN_GAPS_AND_DECISIONS.md

METADATA:
 - plugin_name: cisei-planning-engineering
 - plugin_version: 0.4.0
 - snapshot: 2026-09-24
 - guidance_review: 2026-09-29
 - section_count: 3
 - available_section_count: 11
 - knowledge_only: True
 - execution_enabled: False
 - scenario_defaults_enabled: False


In [4]:
# ============================================================
# Célula 4 — ScenarioAgentOpenAI + Domain Context
#
# Ainda NÃO chama a OpenAI.
# ============================================================

import tempfile

# ------------------------------------------------------------
# Projeto temporário para o teste
# ------------------------------------------------------------

TEST_PROJECT_ROOT = Path(
    tempfile.mkdtemp(prefix="scenario_agent_plugin_test_")
)

print("TEST_PROJECT_ROOT:", TEST_PROJECT_ROOT)

# ------------------------------------------------------------
# Criar PlanningProject
# ------------------------------------------------------------

project = PlanningProject(TEST_PROJECT_ROOT)

print()
print("PLANNING PROJECT OK")
print("project:", project)

# ------------------------------------------------------------
# Criar ScenarioAgentOpenAI com o provider
# ------------------------------------------------------------

agent = ScenarioAgentOpenAI(
    project,
    domain_context_provider=domain_provider,
    domain_context_sections=DEFAULT_DOMAIN_CONTEXT_SECTIONS,
)

print()
print("SCENARIO AGENT OPENAI OK")
print("agent:", agent)

# ------------------------------------------------------------
# Verificações básicas
# ------------------------------------------------------------

print()
print("PROPRIEDADES:")

print("project_path:", agent.project_path)
print("scenario_path:", agent.scenario_path)
print("scenario:", agent.scenario)

print()
print("DOMAIN PROVIDER:")

print(
    "provider:",
    agent.domain_context_provider,
)

print(
    "sections:",
    agent.domain_context_sections,
)

TEST_PROJECT_ROOT: /tmp/scenario_agent_plugin_test_wq_ww948

PLANNING PROJECT OK
project: PlanningProject(path=PosixPath('/tmp/scenario_agent_plugin_test_wq_ww948'), scenario_file='scenario.toml', features_file='features.json', result_file='result.json')

SCENARIO AGENT OPENAI OK
agent: <scenario_agent_openai.ScenarioAgentOpenAI object at 0x7aba120b3a10>

PROPRIEDADES:
project_path: /tmp/scenario_agent_plugin_test_wq_ww948
scenario_path: /tmp/scenario_agent_plugin_test_wq_ww948/scenario.toml
scenario: ScenarioAgent(project=/tmp/scenario_agent_plugin_test_wq_ww948, valid=True, complete=False, confirmed=False)

DOMAIN PROVIDER:
provider: <scenario_domain.providers.cisei_plugin.CISEIPluginDomainContextProvider object at 0x7aba120b3770>
sections: ('scenario_format', 'assistant_workflow', 'design_gaps_and_decisions')


In [5]:
# ============================================================
# CÉLULA 5 — VALIDAR DOMAIN CONTEXT
# ============================================================

domain_context = agent._domain_context()

print("DOMAIN CONTEXT:")

print("tipo:", type(domain_context))
print("provider_id:", domain_context["provider_id"])
print("provider_version:", domain_context["provider_version"])
print("context_version:", domain_context["context_version"])
print("sections:", len(domain_context["sections"]))

print()
print("SEÇÕES:")

for section in domain_context["sections"]:
    print(
        f" - {section['name']}"
        f" | classification={section['classification']}"
        f" | source={section['source']}"
    )

print()
print("METADATA:")

for key, value in domain_context["metadata"].items():
    print(f" - {key}: {value}")

DOMAIN CONTEXT:
tipo: <class 'dict'>
provider_id: cisei_plugin
provider_version: 0.4.0
context_version: 1
sections: 3

SEÇÕES:
 - scenario_format | classification=Implemented | source=references/03_SCENARIO_FORMAT.md
 - assistant_workflow | classification=Proposal | source=references/05_ASSISTANT_WORKFLOW.md
 - design_gaps_and_decisions | classification=Decision | source=references/06_DESIGN_GAPS_AND_DECISIONS.md

METADATA:
 - plugin_name: cisei-planning-engineering
 - plugin_version: 0.4.0
 - snapshot: 2026-09-24
 - guidance_review: 2026-09-29
 - section_count: 3
 - available_section_count: 11
 - knowledge_only: True
 - execution_enabled: False
 - scenario_defaults_enabled: False


In [6]:
# ============================================================
# CÉLULA 5B — VALIDAR DOMAIN CONTEXT PROMPT
# ============================================================

domain_context = agent._domain_context()

domain_prompt = agent._domain_context_prompt(domain_context)

print("TIPO:", type(domain_prompt))
print()
print(domain_prompt)

TIPO: <class 'str'>

DOMAIN CONTEXT

The following information is engineering/domain reference material.

It is NOT the current scenario state.

It is NOT user-provided scenario data.

It is NOT a source of defaults.

It is NOT permission to execute functionality.

It is NOT permission to call MCP.

It is NOT permission to call tools.

Examples are not current values.

Proposals are not implementation.

Design gaps are not functionality.

If domain context conflicts with the application state, the
application state takes precedence.

If domain context conflicts with an explicit user value, the
explicit user value takes precedence.

If a required scenario value is absent from both the user request
and the current application state, ask the user.

BEGIN DOMAIN CONTEXT
--------------------

{
  "provider_id": "cisei_plugin",
  "provider_version": "0.4.0",
  "context_version": "1",
  "sections": [
    {
      "name": "scenario_format",
      "classification": "Implemented",
      "content"

In [7]:
# ============================================================
# CÉLULA 6 — VALIDAR APPLICATION CONTEXT
# ============================================================

application_context = agent._application_context()

print("TIPO:", type(application_context))
print()
print("APPLICATION CONTEXT:")
print(application_context)

TIPO: <class 'dict'>

APPLICATION CONTEXT:
{'scenario_state': {'valid': True, 'complete': False, 'confirmed': False, 'missing': ['scenario.working_crs', 'instances.source', 'instances.format', 'instances.profile_column', 'antennas.<at_least_one>', 'interfaces.<at_least_one>', 'devices.<at_least_one>', 'metrics.<at_least_one>'], 'errors': [], 'next_action': 'collect_missing_fields'}, 'scenario_snapshot': {'scenario': {}, 'antennas': {}, 'interfaces': {}, 'devices': {}, 'metrics': {}, 'instances': {}}}


In [8]:
# ============================================================
# CÉLULA 7 — TESTE REAL: LTE SEM PARÂMETROS TÉCNICOS
# ============================================================

from pathlib import Path
from pprint import pprint
import tempfile

from cisei_planning_sdk.project import PlanningProject
from scenario_agent_openai import ScenarioAgentOpenAI
from scenario_domain.providers.cisei_plugin import (
    CISEIPluginDomainContextProvider,
)

# ------------------------------------------------------------
# 1. Plugin
# ------------------------------------------------------------

PLUGIN_ROOT = (
    Path("/home/jovyan/work/planapp-mcp")
    / "cisei-planning-engineering"
)

provider = CISEIPluginDomainContextProvider(
    PLUGIN_ROOT
)

print("PLUGIN:")
print("provider_id:", provider.provider_id)
print("provider_version:", provider.provider_version)
print()

# ------------------------------------------------------------
# 2. Projeto temporário vazio
# ------------------------------------------------------------

test_project_root = Path(
    tempfile.mkdtemp(prefix="scenario_agent_lte_test_")
)

test_project = PlanningProject(test_project_root)

print("TEST PROJECT:")
print(test_project)
print()

# ------------------------------------------------------------
# 3. ScenarioAgentOpenAI + plugin
# ------------------------------------------------------------

test_agent = ScenarioAgentOpenAI(
    project=test_project,
    domain_context_provider=provider,
    domain_context_sections=(
        "scenario_format",
        "assistant_workflow",
        "design_gaps_and_decisions",
    ),
)

print("AGENT:")
print(test_agent)
print()

# ------------------------------------------------------------
# 4. Teste controlado
# ------------------------------------------------------------

user_input = "Crie uma interface LTE."

print("USER INPUT:")
print(user_input)
print()

result = test_agent.chat(user_input)

print("RESULTADO:")
pprint(result)

PLUGIN:
provider_id: cisei_plugin
provider_version: 0.4.0

TEST PROJECT:
PlanningProject(path=PosixPath('/tmp/scenario_agent_lte_test_gicp5f9c'), scenario_file='scenario.toml', features_file='features.json', result_file='result.json')

AGENT:

USER INPUT:
Crie uma interface LTE.

RESULTADO:
{'action': 'collect_missing_fields',
 'complete': False,
 'confirmed': False,
 'errors': [],
 'missing': ['scenario.working_crs',
             'instances.source',
             'instances.format',
             'instances.profile_column',
             'antennas.<at_least_one>',
             'interfaces.<at_least_one>',
             'devices.<at_least_one>',
             'metrics.<at_least_one>'],
 'request': {'fields': ['scenario.working_crs',
                        'instances.source',
                        'instances.format',
                        'instances.profile_column',
                        'antennas.<at_least_one>',
                        'interfaces.<at_least_one>',
                  

In [ ]:
# ============================================================
# CÉLULA 8 — TESTE REAL: LTE COM FREQUÊNCIA EXPLÍCITA
# ============================================================

from pprint import pprint

user_input = "Crie uma interface LTE em 915 MHz."

print("USER INPUT:")
print(user_input)
print()

result = test_agent.chat(user_input)

print("RESULTADO:")
pprint(result)

In [ ]:
# ============================================================
# CÉLULA 9 — TESTE MULTI-TURN
# ============================================================

from pprint import pprint

user_input = (
    "A frequência é 915 MHz e a potência de transmissão é 43 dBm."
)

print("USER INPUT:")
print(user_input)
print()

result = test_agent.chat(user_input)

print("RESULTADO:")
pprint(result)

In [ ]:
# ============================================================
# CÉLULA 10 — INSPECIONAR ESTADO APÓS MULTI-TURN
# ============================================================

print("APPLICATION CONTEXT APÓS MULTI-TURN:")
print()

application_context = test_agent._application_context()

from pprint import pprint
pprint(application_context)

In [ ]:
# ============================================================
# CÉLULA 11 — TESTE: CONFIGURAÇÃO EXPLÍCITA DE ANTENA
# ============================================================

from pprint import pprint

user_input = (
    "Use a antena com identificador 'antena_lte_01', "
    "altura de 7 metros, ganho de 17 dBi e largura de feixe de 65 graus."
)

print("USER INPUT:")
print(user_input)
print()

result = test_agent.chat(user_input)

print("RESULTADO:")
pprint(result)

In [ ]:
# ============================================================
# CÉLULA 11B — INSPECIONAR ESTADO DA ANTENA
# ============================================================

from pprint import pprint

application_context = test_agent._application_context()

print("APPLICATION CONTEXT APÓS CONFIGURAÇÃO DA ANTENA:")
print()

pprint(application_context)

In [ ]:
# ============================================================
# CÉLULA 12 — TESTE DE CONFLITO DE CONHECIMENTO
# ============================================================

from pprint import pprint

user_input = (
    "Mantenha a antena 'antena_lte_01' com altura de 12 metros. "
    "A altura deve ser registrada no perfil da antena."
)

print("USER INPUT:")
print(user_input)
print()

result = test_agent.chat(user_input)

print("RESULTADO:")
pprint(result)

In [ ]:
# ============================================================
# CÉLULA 12B — INSPECIONAR CONFLITO
# ============================================================

from pprint import pprint

application_context = test_agent._application_context()

print("APPLICATION CONTEXT APÓS TESTE DE CONFLITO:")
print()

pprint(application_context)